# Econ 3916 PS3 Warm-up — the code PS3 asks you to write

**Not graded.** Practice for Problem Set 3, on different data. Each exercise
shows you working code (*Watch*), asks you to write a similar piece yourself
(*Your turn*), then checks it. The exercises follow PS3's order: two tables, a join that fans out, the NULLs a filter drops, a window function, then a confounder and a collider.

- **Turn AI code completion off** (Colab: *Settings › AI assistance*), as PS3
  Phases 1–3 require. You may still click **Explain error**.
- The *Watch* cells use a library (members and their loans) and the *Your turn* cells use a college (students and their course enrollments). The two
  differ **on purpose**: you write each piece yourself instead of copying it.
- Run the cells in order: later exercises use what earlier ones made.
- If a check fails, read its message, fix your cell, and run both again.
- Stuck for more than five minutes? Look back at the *Watch* cell above it.

In [ ]:
import numpy as np
import pandas as pd
import duckdb                          # Colab has it; if this line fails, run !pip install duckdb first
import statsmodels.formula.api as smf

con = duckdb.connect()                 # one in-memory database for the whole warm-up

---
## 1. Two tables, one-to-many (PS3 Step 1.1)

One row per member in `members`, one row per loan in `loans`, linked by `member_id`: that is a one-to-many relation. `np.repeat(ids, counts)` writes each id once per loan. `con.register("name", df)` lets SQL see a DataFrame, and `CREATE TABLE ... AS SELECT * FROM name` copies it into the database as a real table, one you can `UPDATE` later. `con.sql("...")` prints a result; `con.execute("...").fetchone()[0]` hands one number back to Python.

**Watch:** run this and read it.

In [ ]:
rng = np.random.default_rng(7)
n_members = 1_000
n_loans = 1 + rng.poisson(4, size=n_members)                    # every member has borrowed at least once
members_df = pd.DataFrame({
    "member_id": np.arange(1, n_members + 1),
    "branch": rng.choice(["Central", "Riverside", "Hilltop"], size=n_members),
    "fines_paid": np.maximum(2.0 * n_loans + rng.normal(5, 2, n_members), 0).round(2),   # more loans, more fines
})
loans_df = pd.DataFrame({"member_id": np.repeat(members_df["member_id"].to_numpy(), n_loans)})
loans_df.insert(0, "loan_id", np.arange(1, len(loans_df) + 1))

con.register("members_df", members_df)
con.register("loans_df", loans_df)
con.execute("CREATE OR REPLACE TABLE members AS SELECT * FROM members_df")
con.execute("CREATE OR REPLACE TABLE loans AS SELECT * FROM loans_df")
print(con.sql("SELECT COUNT(*) AS members, ROUND(AVG(fines_paid), 2) AS avg_fines FROM members"))
print(con.sql("SELECT COUNT(*) AS loans, COUNT(DISTINCT member_id) AS members_with_loans FROM loans"))

**Your turn:** for 2,000 students, draw `n_courses = rng.integers(1, 9, size=2_000)` (1 to 8 courses each). Build `students_df` with `student_id` (1 to 2,000), `major` drawn from `"Economics"`, `"History"`, `"Biology"`, and `fees = 500 * n_courses + Normal(1000, 300)`, floored at 0. Build `enrollments_df` with one row per course: `enrollment_id` and `student_id`. Register both and create the DuckDB tables `students` and `enrollments`.

In [ ]:
rng = np.random.default_rng(11)
n_students = 2_000
# YOUR CODE HERE
students_df = enrollments_df = None

In [ ]:
assert students_df is not None and len(students_df) == 2_000, "students_df needs one row per student, 2,000 rows"
assert {"student_id", "major", "fees"} <= set(students_df.columns), "students_df needs student_id, major and fees"
assert enrollments_df is not None and len(enrollments_df) > 2_000, "enrollments_df needs one row per COURSE, so more rows than students"
tables = set(con.execute("SHOW TABLES").df()["name"])
assert {"students", "enrollments"} <= tables, "create the DuckDB tables students and enrollments (CREATE TABLE ... AS SELECT)"
n_s, n_e, n_ids = con.execute("""
    SELECT (SELECT COUNT(*) FROM students), (SELECT COUNT(*) FROM enrollments),
           (SELECT COUNT(DISTINCT student_id) FROM enrollments)""").fetchone()
assert n_s == 2_000 and n_e == len(enrollments_df), "the DuckDB tables should hold the same rows as the DataFrames"
assert n_ids == 2_000, "every student takes at least one course, so every student_id appears in enrollments"
r = students_df["fees"].corr(enrollments_df.groupby("student_id").size().reindex(students_df["student_id"]).set_axis(students_df.index))
assert r > 0.7, f"fees should rise with the number of courses (correlation {r:.2f}): fees = 500 * n_courses + noise"
print(f"{n_s:,} students, {n_e:,} enrollments; fees vs courses correlation {r:.2f}")

---
## 2. A join that fans out (PS3 Steps 1.2 and 1.3)

`fines_paid` belongs to the member: one value per member. Join the loans and each member appears once per loan, so an average over the joined rows counts a member with 10 loans ten times. When the column also rises with the number of loans, the average rises with it. Always count the rows an average runs over. A grain check compares `COUNT(*)` with `COUNT(DISTINCT key)` on those rows: more rows than keys means the join fanned out.

**Watch:** run this and read it.

In [ ]:
JOINED_ROWS = "SELECT m.member_id, m.fines_paid FROM members m JOIN loans l ON l.member_id = m.member_id WHERE m.branch = 'Central'"
PLAIN_ROWS = "SELECT member_id, fines_paid FROM members WHERE branch = 'Central'"

per_member, n_plain = con.execute(f"SELECT AVG(fines_paid), COUNT(*) FROM ({PLAIN_ROWS})").fetchone()
per_loan, n_joined = con.execute(f"SELECT AVG(fines_paid), COUNT(*) FROM ({JOINED_ROWS})").fetchone()
print(f"Central, per member: ${per_member:.2f} over {n_plain:,} rows")
print(f"Central, after the join: ${per_loan:.2f} over {n_joined:,} rows")

def member_grain(con, row_sql):
    """A warning string if the rows repeat member_id, else None."""
    n_rows, n_keys = con.execute(f"SELECT COUNT(*), COUNT(DISTINCT member_id) FROM ({row_sql})").fetchone()
    if n_rows > n_keys:
        return f"FAN-OUT: {n_rows:,} rows for {n_keys:,} members"
    return None

print("joined:", member_grain(con, JOINED_ROWS))
print("plain: ", member_grain(con, PLAIN_ROWS))

**Your turn:** for **Economics** students, compute the average `fees` per student as `avg_per_student`, then the average after joining `enrollments` as `avg_after_join`, and the number of rows that second average runs over as `rows_after_join`. Then write `rows_check(con, row_sql, key)` that returns a warning string when the rows repeat `key` and `None` when they don't, and run it on your two row queries: `warn_joined` and `warn_plain`.

In [ ]:
# YOUR CODE HERE
avg_per_student = avg_after_join = rows_after_join = None

def rows_check(con, row_sql, key):
    pass

warn_joined = warn_plain = None

In [ ]:
econ = students_df["major"] == "Economics"
assert avg_per_student is not None and abs(avg_per_student - students_df.loc[econ, "fees"].mean()) < 0.01, \
    "avg_per_student: the plain average of fees over Economics students, one row each"
expected_rows = enrollments_df["student_id"].isin(students_df.loc[econ, "student_id"]).sum()
assert rows_after_join == expected_rows, f"the joined query should run over {expected_rows:,} rows, one per Economics enrollment"
assert avg_after_join > 1.1 * avg_per_student, "the joined average should sit well above the per-student one: did you join enrollments?"
assert isinstance(warn_joined, str), "rows_check should return a warning string for the joined rows"
assert warn_plain is None, "rows_check should return None for the plain query: one row per student"
print(f"per student ${avg_per_student:,.2f} | after the join ${avg_after_join:,.2f} over {rows_after_join:,} rows")
print(f"check on the joined rows: {warn_joined}")
print("Same column, same students: the join changed the unit from a student to an enrollment.")

---
## 3. The rows a filter drops (PS3 Step 2.2)

NULL means *unknown*. For a member whose branch is NULL, `branch != 'Central'` is not true and not false: it is NULL (unknown). A `WHERE` clause keeps only the rows where its condition is **true**, so those members are **dropped**, with no error and no warning. To keep them, say so: `branch IS DISTINCT FROM 'Central'` treats NULL as a value that differs from 'Central', and `branch != 'Central' OR branch IS NULL` does the same in older SQL.

**Watch:** run this and read it.

In [ ]:
unknown_ids = rng.choice(members_df["member_id"].to_numpy(), size=100, replace=False)   # 10% of members
con.register("unknown_ids", pd.DataFrame({"member_id": unknown_ids}))
con.execute("UPDATE members SET branch = NULL WHERE member_id IN (SELECT member_id FROM unknown_ids)")

n_ne = con.execute("SELECT COUNT(*) FROM members WHERE branch != 'Central'").fetchone()[0]
n_distinct = con.execute("SELECT COUNT(*) FROM members WHERE branch IS DISTINCT FROM 'Central'").fetchone()[0]
print(f"!= 'Central': {n_ne:,} rows | IS DISTINCT FROM 'Central': {n_distinct:,} rows | dropped by !=: {n_distinct - n_ne:,}")
print(con.sql("SELECT NULL != 'Central' AS null_ne, NULL IS DISTINCT FROM 'Central' AS null_distinct"))

**Your turn:** 400 students (20%) have not declared a major. Pick them with `rng.choice(..., replace=False)` and set their `major` to NULL in the `students` table with an `UPDATE`. Then count the NULLs as `n_null`, the rows `WHERE major != 'Economics'` returns as `n_dropped_query`, and the rows of a query that keeps the undeclared students as `n_kept_query`.

In [ ]:
# YOUR CODE HERE
n_null = n_dropped_query = n_kept_query = None

In [ ]:
db_nulls = con.execute("SELECT COUNT(*) FROM students WHERE major IS NULL").fetchone()[0]
assert n_null is not None and n_null == 400, "set exactly 400 majors to NULL and count them with IS NULL"
assert db_nulls == 400, "the NULLs have to be in the DuckDB table students: use UPDATE ... SET major = NULL"
assert n_dropped_query is not None and n_kept_query is not None, "count both queries"
assert n_kept_query - n_dropped_query == n_null, "the query that keeps the unknowns should return exactly n_null more rows"
print(f"!= 'Economics': {n_dropped_query:,} | keeping the undeclared: {n_kept_query:,} | difference {n_kept_query - n_dropped_query:,} = the NULLs")

---
## 4. A window function (PS3 Step 2.3)

`GROUP BY branch` collapses the table to one row per branch. `AVG(fines_paid) OVER (PARTITION BY branch)` computes the same averages but writes each one on every row of its branch, so every member keeps their own row and gains a column. Rows with a NULL branch form a partition of their own.

**Watch:** run this and read it.

In [ ]:
WATCH_SQL = """
    SELECT member_id, branch, fines_paid,
           AVG(fines_paid) OVER (PARTITION BY branch) AS branch_avg
    FROM members
    ORDER BY member_id
"""
watch_df = con.execute(WATCH_SQL).df()
print(watch_df.head(6).round(2).to_string(index=False))
print(f"rows in {n_members:,}, rows out {len(watch_df):,}")

**Your turn:** in **one** query with no join and no subquery, return each student's `student_id`, `major`, `fees`, their major's average as `major_avg`, and `fees` minus that average as `vs_major`. Put the SQL text in `window_sql` and the result in the DataFrame `with_avg`.

In [ ]:
# YOUR CODE HERE
window_sql = ""
with_avg = None

In [ ]:
assert with_avg is not None and len(with_avg) == 2_000, "one row per student: a window function never changes the row count"
assert {"major_avg", "vs_major"} <= set(with_avg.columns), "name the new columns major_avg and vs_major"
sql_up = window_sql.upper()
assert " JOIN " not in sql_up and sql_up.count("SELECT") == 1, "one query: no JOIN and no subquery"
assert "OVER" in sql_up, "use a window function: AVG(...) OVER (PARTITION BY ...)"
group_mean = with_avg.groupby("major", dropna=False)["fees"].transform("mean")
assert np.allclose(with_avg["major_avg"], group_mean), "major_avg should be the average fees of the student's own major"
assert np.allclose(with_avg["vs_major"], with_avg["fees"] - with_avg["major_avg"]), "vs_major = fees - major_avg"
print(with_avg.head(6).round(2).to_string(index=False))
print(f"{(with_avg['vs_major'] > 0).sum():,} students pay more than their major's average")

---
## 5. A confounder, then a collider (PS3 Phase 3)

Simulate a world where you set the true effect, so you can score each estimate against it. A **confounder** drives both who gets the treatment and the outcome: the raw gap mixes the two, and comparing like with like (adding it as a control) recovers the truth. A variable that is **caused by** the treatment and by the outcome works the other way: holding it fixed compares unlike units, and the estimate moves away from the truth. `smf.ols("y ~ d + x", data=df).fit().params["d"]` is the estimate on `d` holding `x` fixed.

**Watch:** run this and read it.

In [ ]:
TRUE_CLUB = 2.0                                   # the book club adds 2 loans a year: the truth on file
n_people = 5_000
free_hours = rng.normal(10, 3, n_people)           # the confounder
book_club = (rng.random(n_people) < 1 / (1 + np.exp(-(free_hours - 10) / 2))).astype(int)   # more free time, more joining
loans_year = 5 + 1.5 * free_hours + TRUE_CLUB * book_club + rng.normal(0, 3, n_people)
events = rng.poisson(1 + 3 * book_club + 0.3 * loans_year)     # caused by the club AND by borrowing
lib = pd.DataFrame({"free_hours": free_hours, "book_club": book_club, "loans_year": loans_year, "events": events})

raw = lib.loc[lib.book_club == 1, "loans_year"].mean() - lib.loc[lib.book_club == 0, "loans_year"].mean()
held = smf.ols("loans_year ~ book_club + free_hours", data=lib).fit().params["book_club"]
worse = smf.ols("loans_year ~ book_club + free_hours + events", data=lib).fit().params["book_club"]
print(f"truth {TRUE_CLUB:.2f} | raw gap {raw:.2f} | holding free hours {held:.2f} | adding events {worse:.2f}")

**Your turn:** 4,000 students. `prior_gpa` is Normal(3.0, 0.4). Tutoring goes to students who were struggling: `tutored` is 1 with probability `1 / (1 + np.exp((prior_gpa - 3.0) / 0.2))`. The exam score is `50 + 10 * prior_gpa + 3 * tutored + Normal(0, 5)`, so the true effect of tutoring is **+3 points**. `badges` (earned on the course site) is Poisson with mean `np.maximum(0, 10 + 6 * tutored + 0.6 * (exam - 80))`: tutors assign badge tasks, and students with higher scores earn more. Put them in a DataFrame and estimate the effect three ways: the raw gap `naive`, holding prior GPA fixed `adjusted`, and holding prior GPA and badges fixed `with_badges`.

In [ ]:
TRUE_EFFECT = 3.0
n_kids = 4_000
# YOUR CODE HERE
naive = adjusted = with_badges = None

In [ ]:
assert naive is not None and naive < TRUE_EFFECT - 2, \
    "tutored students started with lower GPAs, so the raw gap should sit well below the true +3"
assert adjusted is not None and abs(adjusted - TRUE_EFFECT) < 1.0, "holding prior_gpa fixed should land within a point of +3"
assert with_badges is not None and with_badges < adjusted - 1.0, \
    "adding badges should pull the estimate further from the truth: badges are caused by tutoring AND by the exam score"
print(f"truth {TRUE_EFFECT:+.2f} | naive {naive:+.2f} | adjusted {adjusted:+.2f} | with badges {with_badges:+.2f}")
print("The confounder belongs in the comparison; something the treatment and the outcome both cause does not.")

---
**Done.** Two linked tables, a join that fans out and the check that catches it, the rows a NULL filter drops, a window function, and a confounder next to a collider: the building blocks of PS3 Phases 1–3. Now open `ps3-starter.ipynb`.